In [ ]:
import pandas as pd
from joblib import load, dump
import matplotlib.pyplot as plt

#from glotlid.fidelity import languages_iso_input
%matplotlib inline
import numpy as np

import matplotlib.cm as cm
from random import shuffle, seed
from utils.missing_data import dico_other_iso_to_genus

In [ ]:
dico_other_iso_to_genus

# Load the results of MultiQ for a given model

In [ ]:
name_model = "Qwen1.5-7B-Chat"

path_multiq: str = f"../multiQ/data/language_fidelity/{name_model}.csv"
df_multiq: pd.DataFrame = pd.read_csv(path_multiq)

In [ ]:
if "gpt-4_evaluation" in df_multiq.columns:
    df_switch = df_multiq.rename(columns={"gpt-4_evaluation": "eval_completion"})

In [ ]:
df_multiq

# Load maps

In [ ]:
maps_wals2genus: dict[str, str] = load("../multiQ/data/maps/mapping_wals_genus.joblib")
maps_iso2wals: dict[str, str] = load("../multiQ/data/maps/iso_to_wals.joblib")

# Map languages to their genus

### Create a dictionary of genus for each input language

In [ ]:
list_languages_iso_input = list(df_multiq["iso_639_3"].unique())
dico_genus_inputs = {}
for lg_input in list_languages_iso_input:
    if lg_input in maps_iso2wals:
        wals_lg_input = maps_iso2wals[lg_input]
        if type(wals_lg_input) == str:
            if wals_lg_input in maps_wals2genus:
                lg_genus = maps_wals2genus[wals_lg_input]
            else:
                lplp
                print(lg_input)
        else:
            if maps_wals2genus[wals_lg_input[0]] == maps_wals2genus[wals_lg_input[1]]:
                lg_genus = maps_wals2genus[wals_lg_input[0]]
            else:
                print("list of wals")
                print(lg_input, wals_lg_input)
                lplp

    else:
        lg_genus = dico_other_iso_to_genus[lg_input]

    dico_genus_inputs[lg_input] = lg_genus
print(len(list_languages_iso_input))
print(len(dico_genus_inputs))

## Create a dictionary of genus for each output language

In [ ]:
list_languages_iso_output = list(df_multiq["detected_language"].unique())
dico_genus_outputs = {}
for lg_output in list_languages_iso_output:
    if lg_output in maps_iso2wals:
        wals_lg_output = maps_iso2wals[lg_output]
        if type(wals_lg_output) == str:
            if wals_lg_output in maps_wals2genus:
                lg_genus = maps_wals2genus[wals_lg_output]
            else:
                print(lg_output)
                lplp
        else:
            if maps_wals2genus[wals_lg_output[0]] == maps_wals2genus[wals_lg_output[1]]:
                lg_genus = maps_wals2genus[wals_lg_output[0]]
            else:
                print("list of wals")
                print(lg_output, wals_lg_output)
                lplp

    else:
        if lg_output not in dico_other_iso_to_genus:

            #print(f"'{lg_output}' : '{suppl_map_iso2genus[lg_output]}', ")
            print(f"'{lg_output}'")
        elif lg_output not in [
        "rop",
            "nzi"
        ]:
            lg_genus = dico_other_iso_to_genus[lg_output]

    dico_genus_outputs[lg_output] = lg_genus

 # Calculate fidelity

In [ ]:
occurrences_input_genus = {}
occurrences_output_genus_by_input_genus = {}

In [ ]:
for lg_input in list_languages_iso_input:
    genus_input = dico_genus_inputs[lg_input]
    df_input = df_multiq[df_multiq["iso_639_3"] == lg_input]
    occurrences_input_genus[genus_input] = occurrences_input_genus.get(genus_input, 0) + len(df_input)
    if genus_input not in occurrences_output_genus_by_input_genus:
        occurrences_output_genus_by_input_genus[genus_input] = {}
    for lg_output in df_input["detected_language"]:
        genus_output = dico_genus_outputs[lg_output]
        occurrences_output_genus_by_input_genus[genus_input][genus_output] = occurrences_output_genus_by_input_genus[genus_input].get(genus_output, 0) + 1

In [ ]:
repartition_answers_genus_final = {}
for genus_input in occurrences_output_genus_by_input_genus:
    total_occurrences = occurrences_input_genus[genus_input]
    repartition_answers_genus = {}
    for genus_output in occurrences_output_genus_by_input_genus[genus_input]:
        count = occurrences_output_genus_by_input_genus[genus_input][genus_output]
        repartition_answers_genus[genus_output] = count / total_occurrences
    repartition_answers_genus_final[genus_input] = repartition_answers_genus

In [ ]:
repartition_answers_genus_final

In [ ]:
#sum all values < 0.01 into "other"
for genus_input in repartition_answers_genus_final:
    repart_genus = repartition_answers_genus_final[genus_input]
    other = 0
    for genus_output in list(repart_genus.keys()):
        if repart_genus[genus_output] < 0.05:
            other += repart_genus[genus_output]
            del repart_genus[genus_output]
    if other > 0:
        repart_genus["Other"] = other
    repartition_answers_genus_final[genus_input] = repart_genus

In [ ]:
# calculate fidelity score :  # sum of the diagonal values (where input genus = output genus), weighted by the number of occurrences of each input genus
fidelity_score = 0
for genus_input in repartition_answers_genus_final:
    repart_genus = repartition_answers_genus_final[genus_input]
    if genus_input in repart_genus:
        fidelity_score += repart_genus[genus_input] * occurrences_input_genus[genus_input]
fidelity_score = fidelity_score / len(df_multiq)
print(f"Fidelity score for model {name_model} : {fidelity_score:.4f}")

# Plot results

In [ ]:
import os
if not os.path.exists(f"subfamilies/results/{name_model}"):
    os.makedirs(f"subfamilies/results/{name_model}")

if not os.path.exists(f"utils"):
    os.makedirs(f"utils")

In [ ]:
seed(1)
cmap = cm.get_cmap('gist_ncar', 200)  # 200 discrete colors
# Generate a list of 200 RGBA colors
colors_list = [cmap(i) for i in range(cmap.N)]
shuffle(colors_list)

In [ ]:
#maps_colors_gen : dict[str, tuple]= load("utils/mapping_gen_colors.joblib")
maps_colors_gen = {}

for genus_source_plot in repartition_answers_genus_final:
    repart_genus = repartition_answers_genus_final[genus_source_plot]


    labels = []
    sizes = []
    colors = []
    other = 0
    for target_genus in repart_genus:
        labels.append(target_genus)
        sizes.append(repart_genus[target_genus])
        if target_genus not in maps_colors_gen:
            maps_colors_gen[target_genus] = colors_list[len(maps_colors_gen)]
        colors.append(maps_colors_gen[target_genus])


    if other > 0:
        labels.append("Other")
        sizes.append(other)
        if "Other" not in maps_colors_gen:
                maps_colors_gen["Other"] = colors_list[len(maps_colors_gen)]
        colors.append(maps_colors_gen["Other"])

    dump(maps_colors_gen, "utils/mapping_gen_colors.joblib")


    explode = [0.1 if size == max(sizes) else 0 for size in sizes]

    plt.clf()

    plt.pie(sizes, labels=labels,explode=explode, colors=colors,
            autopct='%1.1f%%', shadow=True, startangle=140)

    plt.axis('equal')  # Equal aspect ratio ensures the pie is a circle
    plt.title(f"Model : {name_model} - Genus : {genus_source_plot}")
    #plt.show()
    plt.savefig(f"subfamilies/results/{name_model}/pie_gen_{genus_source_plot}.png", bbox_inches='tight')